# QuickBite Delivery Intelligence

This notebook reproduces the synthetic data pipeline used for the presentation deck.

## 1. Setup

Load libraries, generate sample order data, and save CSV/SQLite outputs.

In [ ]:
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 2. Explore the data

The dataset contains city, cuisine, traffic, weather, delivery time, rating, and late-delivery labels.

In [ ]:
df = pd.read_csv('data/orders.csv')
df.head()

## 3. Summary statistics

In [ ]:
df[['order_value','delivery_time_min','customer_rating']].describe().T

## 4. Train a classifier

In [ ]:
features = ['order_hour','city','restaurant_type','cuisine','channel','payment_mode','traffic_level','weather','distance_km','prep_time_min','order_value','weekend']
target = 'late_delivery'
cat_cols = ['city','restaurant_type','cuisine','channel','payment_mode','traffic_level','weather']
num_cols = ['order_hour','distance_km','prep_time_min','order_value','weekend']
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=42, stratify=y)

pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])),
    ('model', RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced_subsample', min_samples_leaf=3))
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:, 1]
print('Accuracy:', accuracy_score(y_test, pred))
print('Precision:', precision_score(y_test, pred))
print('Recall:', recall_score(y_test, pred))
print('F1:', f1_score(y_test, pred))
print('ROC-AUC:', roc_auc_score(y_test, proba))